In [0]:
spark.sql("DROP SCHEMA IF EXISTS workspace.loan_bronze CASCADE")  # CASCADE drops all tables inside
spark.sql("DROP SCHEMA IF EXISTS workspace.loan_silver CASCADE")
spark.sql("DROP SCHEMA IF EXISTS workspace.loan_gold   CASCADE")

print("✅ Old schemas dropped")

# Now recreate fresh
spark.sql("CREATE SCHEMA workspace.loan_bronze COMMENT 'Raw loan events'")
spark.sql("CREATE SCHEMA workspace.loan_silver COMMENT 'Cleaned and classified loans'")
spark.sql("CREATE SCHEMA workspace.loan_gold   COMMENT 'Aggregated reports and metrics'")

print("✅ Fresh schemas created")

In [0]:
spark.sql("""
        CREATE TABLE IF NOT EXISTS loan_bronze.raw_loan_events (
            event_id STRING,
            event_type STRING,
            facility_id BIGINT,
            buyer_pan STRING,
            fi_name STRING,
            amount DOUBLE,
            due_date DATE,
            event_timestamp TIMESTAMP,
            raw_payload STRING,
            ingested_at TIMESTAMP
        )
USING DELTA
PARTITIONED BY (event_type)
""")

print("Table created")



In [0]:
spark.sql("""
        CREATE TABLE IF NOT EXISTS loan_silver.classified_loans (
            facility_id BIGINT,
            rff_id BIGINT,
            buyer_pan STRING,
            fi_name STRING,
            is_colending BOOLEAN,
            colending_share DOUBLE,
            total_receivable DOUBLE,
            principal_receivable DOUBLE,
            due_date DATE,
            days_overdue INTEGER,
            dpd_bucket STRING,
            is_npa BOOLEAN,
            npa_date DATE,
            processing_date DATE,
            updated_at TIMESTAMP
        )
USING DELTA
PARTITIONED BY (processing_date)
""")

print("Table created")

In [0]:
spark.sql("""
          ALTER TABLE loan_silver.classified_loans
          SET TBLPROPERTIES(
              'delta.enableChangeDataFeed' = 'true',
              'delta.autoOptimize.optimizeWrite' = 'true',
              'delta.autoOptimize.autoCompact' = 'true'
          )
""")


# delta.enableChangeDataFeed = 'true' — activates the Change Data Feed (CDF). Every write to the table produces a hidden _change_data log recording which rows were inserted, updated, or deleted, along with the Delta version. Downstream jobs can query readChangeFeed = true to consume only the diff since their last checkpoint — essential for Silver→Gold incremental pipelines and CDC use cases.


# delta.autoOptimize.optimizeWrite = 'true' — fixes the small files problem at write time. Spark reshuffles output before writing so each Parquet file is ~128 MB, rather than producing one tiny file per partition or task. This is especially valuable for streaming jobs that write small micro-batches frequently.


# delta.autoOptimize.autoCompact = 'true' — fixes small files after the fact. After each commit, Delta checks the file count; if it crosses a threshold, it runs a background OPTIMIZE to merge small files into larger ones. Think of it as a cleanup crew that runs asynchronously without blocking writers. It complements optimizeWrite rather than replacing it — together they give you clean file layouts at both write time and over time.


# For a loan table like classified_loans, all three together mean: downstream risk models see only new/changed loans (CDF), read scans stay fast (no small file penalty), and the table stays tidy over months of incremental loads without manual OPTIMIZE runs.

In [0]:
spark.read.format("delta").option("readChangeFeed","true").option("startingVersion",0).table("loan_silver.classified_loans").display()

In [0]:
spark.sql("""
        CREATE TABLE IF NOT EXISTS loan_gold.aging_report (
            buyer_pan STRING,
            dpd_bucket              STRING,
            total_receivable_0_90   DOUBLE,
            total_receivable_90plus DOUBLE,
            bucket_0_7_days         DOUBLE,
            bucket_8_14_days        DOUBLE,
            bucket_15_30_days       DOUBLE,
            bucket_1_2_months       DOUBLE,
            bucket_2_3_months       DOUBLE,
            bucket_3_6_months       DOUBLE,
            bucket_6_12_months      DOUBLE,
            bucket_1_3_years        DOUBLE,
            as_on_date              DATE,
            report_generated_at     TIMESTAMP
        )
    USING DELTA    
    """)
print("✅ All Delta tables created")

In [0]:
import random
import builtins
from datetime import datetime, timedelta
from pyspark.sql.types import *



BUYERS = [
    {"pan": "AAAPZ1234C", "name": "Business A"},
    {"pan": "BBBPZ5678D", "name": "Business B"},
    {"pan": "CCCPZ9999E", "name": "Business C"},
    {"pan": "DDDPZ1111F", "name": "Business D"},
    {"pan": "EEEEZ2222G", "name": "Business E"},
]

LENDERS = [
    {"fi_name": "Progfin",      "is_colending": False, "share": 1.0},
    {"fi_name": "Progfin-HDFC", "is_colending": True,  "share": 0.2},
    {"fi_name": "Progfin-SBI",  "is_colending": True,  "share": 0.2},
    {"fi_name": "Progfin-ICICI", "is_colending": True,  "share": 0.2},
    {"fi_name": "Progfin-AXIS",  "is_colending": True,  "share": 0.2}
]

EVENT_TYPES = ["DISBURSEMENT", "REPAYMENT", "OVERDUE", "NPA_FLAGGED"]

def generate_loan_event():
    buyer = random.choice(BUYERS)
    lender = random.choice(LENDERS)

    # Random due date between 90 days ago and 6 months from now
    days_offset = random.randint(-90, 180)
    due_date = (datetime.now() + timedelta(days=days_offset)).date()

    event = {
        "event_id":  str(random.randint(100000, 999999)),
        "event_type": random.choice(EVENT_TYPES),
        "facility_id": random.randint(10000, 99999),
        "buyer_pan": buyer["pan"],
        "fi_name": lender["fi_name"],
        "amount": builtins.float(builtins.round(random.uniform(100000, 5000000), 2)),
        "due_date": str(due_date),
        "event_timestamp": datetime.now().isoformat(),
    }
    return event

def write_events_to_landing_zone(num_events = 500):
    events = [generate_loan_event() for _ in range (num_events)]

    schema = StructType([
        StructField("event_id",        StringType()),
        StructField("event_type",      StringType()),
        StructField("facility_id",     LongType()),
        StructField("buyer_pan",       StringType()),
        StructField("fi_name",         StringType()),
        StructField("amount",          DoubleType()),
        StructField("due_date",        StringType()),
        StructField("event_timestamp", StringType()),
    ])

    df = spark.createDataFrame(events, schema)
    
    # Write as JSON files to landing zone
    # Auto Loader monitors this path
   
    df.write.mode("append").json("/Volumes/dgs/default/cricket_api_project/loan_events/")

    print(f"✅ Written {num_events} events to landing zone")

    return df

write_events_to_landing_zone()


In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *


raw_schema = StructType([
    StructField("event_id",        StringType(),  True),
    StructField("event_type",      StringType(),  True),
    StructField("facility_id",     LongType(),    True),
    StructField("buyer_pan",       StringType(),  True),
    StructField("fi_name",         StringType(),  True),
    StructField("amount",          DoubleType(),  True),
    StructField("due_date",        StringType(),  True),
    StructField("event_timestamp", StringType(),  True),
])

raw_stream = (
    spark.readStream
    .format("cloudFiles")               # Auto Loader format
    .option("cloudFiles.format", "json") # Source file format
    .option("cloudFiles.schemaLocation", # Where to store inferred schema
            "/Volumes/dgs/default/cricket_api_project/loan_events/loan_monitoring/schema/bronze/")
    .option("cloudFiles.inferColumnTypes", "true")
    .schema(raw_schema)
    .load("/Volumes/dgs/default/cricket_api_project/loan_events/")  # Landing zone path
)

bronze_df = (
    raw_stream
    .withColumn("ingested_at",  current_timestamp())
    .withColumn("raw_payload",  to_json(struct("*")))  # Store full raw payload
    .withColumn("event_timestamp", 
                col("event_timestamp").cast(TimestampType()))
    .withColumn("due_date",     
                col("due_date").cast(DateType()))
)


bronze_query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")               # Always append raw data
    .option("checkpointLocation",       # Checkpoint = fault tolerance
            "/Volumes/dgs/default/cricket_api_project/loan_events/loan_monitoring/checkpoints/bronze/")
    .option("mergeSchema", "true")      # Handle schema evolution
    .trigger(availableNow=True)         # Process all available data and stop
    .table("loan_bronze.raw_loan_events")
)

print("✅ Bronze streaming pipeline started")
print(f"   Status: {bronze_query.status}")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable


def get_dpd_bucket(days_overdue, is_npa):
    """
    Assigns DPD bucket based on overdue days
    NPA always overrides to 90+
    """
    if is_npa:
        return "90+"
    elif days_overdue is None or days_overdue <= 0:
        return "Non Overdue"
    elif days_overdue <= 30:
        return "0-30"
    elif days_overdue <= 60:
        return "31-60"
    elif days_overdue <= 90:
        return "61-90"
    else:
        return "90+"
    
dpd_bucket_udf = udf(get_dpd_bucket, StringType())

colending_lookup = spark.createDataFrame([
    ("Progfin",      False, 1.0),
    ("Progfin-HDFC", True,  0.2),
    ("Progfin-SBI",  True,  0.2),
    ("Progfin-ICICI", True, 0.2),
    ("Progfin-AXIS",  True,  0.2)
], ["fi_name", "is_colending", "colending_share"])

bronze_stream = (
    spark.readStream
    .format("delta")
    .table("loan_bronze.raw_loan_events")
)


silver_df = (
    bronze_stream
    .join(colending_lookup, on = 'fi_name', how = 'left')
    .withColumn("days_overdue", datediff(current_date(), col("due_date")))
    .withColumn("adjusted_amount", col("amount") * col("colending_share"))
    .withColumn("is_npa", col("event_type") == "NPA_FLAGGED")
    .withColumn("dpd_bucket", 
                   dpd_bucket_udf(
                       col("days_overdue"),
                       col("is_npa")
                   )
                )
    .withColumn("npa_date", when(col("is_npa"), current_date()).otherwise(lit(None).cast("date")))
    .withColumn("processing_date", current_date())
    .withColumn("updated_at", current_timestamp())
    .select(
        "facility_id",
        lit(None).cast("bigint").alias("rff_id"),
        "buyer_pan",
        "fi_name",
        "is_colending",
        "colending_share",
        col("adjusted_amount").alias("total_receivable"),
        col("amount").alias("principal_receivable"),
        "due_date",
        "days_overdue",
        "dpd_bucket",
        "is_npa",
        "npa_date",
        "processing_date",
        "updated_at"
    )
)


def upsert_to_silver(batch_df, batch_id):
    """
    Called for every micro-batch from the stream
    batch_df = current batch of records
    batch_id = unique ID for this batch
    """
    
    # Get reference to Silver Delta table
    deduped_df = batch_df.dropDuplicates(["facility_id", "processing_date"])
    silver_table = DeltaTable.forName(spark, "loan_silver.classified_loans")
    
    # MERGE logic:
    #   If facility_id + processing_date matches → UPDATE
    #   If no match → INSERT new record
    (
        silver_table.alias("silver")
        .merge(
            deduped_df.alias("updates"),
            """
            silver.facility_id      = updates.facility_id AND
            silver.processing_date  = updates.processing_date
            """
        )
        .whenMatchedUpdate(set={
            "total_receivable":     "updates.total_receivable",
            "dpd_bucket":           "updates.dpd_bucket",
            "is_npa":               "updates.is_npa",
            "npa_date":             "updates.npa_date",
            "days_overdue":         "updates.days_overdue",
            "updated_at":           "updates.updated_at"
        })
        .whenNotMatchedInsert(values={
            "facility_id":          "updates.facility_id",
            "rff_id":               "updates.rff_id",
            "buyer_pan":            "updates.buyer_pan",
            "fi_name":              "updates.fi_name",
            "is_colending":         "updates.is_colending",
            "colending_share":      "updates.colending_share",
            "total_receivable":     "updates.total_receivable",
            "principal_receivable": "updates.principal_receivable",
            "due_date":             "updates.due_date",
            "days_overdue":         "updates.days_overdue",
            "dpd_bucket":           "updates.dpd_bucket",
            "is_npa":               "updates.is_npa",
            "npa_date":             "updates.npa_date",
            "processing_date":      "updates.processing_date",
            "updated_at":           "updates.updated_at"
        })
        .execute()
    )
    
    print(f"✅ Batch {batch_id} merged into Silver table")

# ─────────────────────────────────────────
# STEP 6: Start Silver Streaming Pipeline
# foreachBatch = run our upsert function
# on every micro-batch
# ─────────────────────────────────────────

silver_query = (
    silver_df.writeStream
    .format("delta")
    .foreachBatch(upsert_to_silver)    # Call our merge function per batch
    .option("checkpointLocation",
            "/Volumes/dgs/default/cricket_api_project/loan_events/loan_monitoring/checkpoints/silver/")
    .trigger(availableNow=True) # Process every 1 minute
    .start()
)

print("✅ Silver streaming pipeline started")

In [0]:
# 05_gold/01_aging_report.py
# PURPOSE: Generate the aging report from Silver layer
#          Same logic as your original SQL report
#          But now in PySpark
#
# KEY CONCEPTS:
#   - Spark SQL on Delta tables 
#   - Window aggregations
#   - Z-Ordering for query optimization
#   - Scheduled batch processing

from pyspark.sql.functions import *
from delta.tables import DeltaTable

# ─────────────────────────────────────────
# STEP 1: Read from Silver Layer
# ─────────────────────────────────────────

silver_df = spark.table("loan_silver.classified_loans")

# ─────────────────────────────────────────
# STEP 2: Build Aging Report
# Same logic as your SQL report
# ─────────────────────────────────────────

current_month_start = date_trunc("month", current_date())

aging_df = (
    silver_df
    .groupBy("buyer_pan", "dpd_bucket")
    .agg(

        # Total 0-90 (non NPA)
        sum(when(col("dpd_bucket") != "90+",
                 col("total_receivable"))
           ).alias("total_receivable_0_90"),

        # Total 90+ (NPA — principal only)
        sum(when(col("dpd_bucket") == "90+",
                 col("principal_receivable"))
           ).alias("total_receivable_90plus"),

        # Bucket 1: 0-7 days
        sum(when(
            (col("dpd_bucket") != "90+") &
            (col("due_date").between(
                current_month_start,
                date_add(current_month_start, 6)
            )),
            col("total_receivable")
        )).alias("bucket_0_7_days"),

        # Bucket 2: 8-14 days
        sum(when(
            (col("dpd_bucket") != "90+") &
            (col("due_date").between(
                date_add(current_month_start, 7),
                date_add(current_month_start, 13)
            )),
            col("total_receivable")
        )).alias("bucket_8_14_days"),

        # Bucket 3: 15-30 days
        sum(when(
            (col("dpd_bucket") != "90+") &
            (col("due_date").between(
                date_add(current_month_start, 14),
                last_day(current_month_start)
            )),
            col("total_receivable")
        )).alias("bucket_15_30_days"),

        # Bucket 4: 1-2 months
        sum(when(
            (col("dpd_bucket") != "90+") &
            (col("due_date").between(
                add_months(current_month_start, 1),
                date_add(add_months(current_month_start, 2), -1)
            )),
            col("total_receivable")
        )).alias("bucket_1_2_months"),

        # Bucket 5: 2-3 months
        sum(when(
            (col("dpd_bucket") != "90+") &
            (col("due_date").between(
                add_months(current_month_start, 2),
                date_add(add_months(current_month_start, 3), -1)
            )),
            col("total_receivable")
        )).alias("bucket_2_3_months"),

        # Bucket 6: 3-6 months + already overdue catch-all
        sum(when(
            (col("dpd_bucket") != "90+") &
            (
                # Future 3-6 months
                col("due_date").between(
                    add_months(current_month_start, 3),
                    date_add(add_months(current_month_start, 6), -1)
                ) |
                # Already overdue catch-all
                (col("due_date") < current_month_start)
            ),
            col("total_receivable")
        )).alias("bucket_3_6_months"),

        # Bucket 7: 6-12 months
        sum(when(
            (col("dpd_bucket") != "90+") &
            (col("due_date").between(
                add_months(current_month_start, 6),
                date_add(add_months(current_month_start, 12), -1)
            )),
            col("total_receivable")
        )).alias("bucket_6_12_months"),

        # Bucket 8: 1-3 years
        sum(when(
            (col("dpd_bucket") != "90+") &
            (col("due_date").between(
                add_months(current_month_start, 12),
                date_add(add_months(current_month_start, 49), -1)
            )),
            col("total_receivable")
        )).alias("bucket_1_3_years"),
    )
    .withColumn("as_on_date",
        date_add(current_month_start, -1))
    .withColumn("report_generated_at",
        current_timestamp())

    # Convert to Crores (divide by 10 million)
    .withColumn("total_receivable_0_90",
        round(col("total_receivable_0_90") / 1e7, 2))
    .withColumn("total_receivable_90plus",
        round(col("total_receivable_90plus") / 1e7, 2))
)

# ─────────────────────────────────────────
# STEP 3: Write to Gold Delta Table
# Overwrite each time — this is a full refresh report
# ─────────────────────────────────────────

(
    aging_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("loan_gold.aging_report")
)

print("✅ Gold aging report refreshed")

# ─────────────────────────────────────────
# STEP 4: Optimize Gold Table with Z-Ordering
# Z-Order clusters data by buyer_pan and dpd_bucket
# Makes queries on these columns much faster
# ─────────────────────────────────────────

spark.sql("""
    OPTIMIZE loan_gold.aging_report
    ZORDER BY (buyer_pan, dpd_bucket)
""")

print("✅ Gold table optimized with Z-Ordering")

In [0]:
# 06_alerts/01_high_risk_alerts.py
# PURPOSE: Detect high risk loans in real time
#          Alert when buyer crosses DPD thresholds
#
# KEY CONCEPTS:
#   - Streaming aggregations
#   - foreachBatch for custom alert logic
#   - Delta Change Data Feed (CDF)

from pyspark.sql.functions import *
from delta.tables import DeltaTable

# ─────────────────────────────────────────
# STEP 1: Read Change Data Feed from Silver
# CDF tracks what changed — only process changes
# Not the full table every time
# ─────────────────────────────────────────

changes_df = (
    spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")   # Read only changes
    .option("startingVersion", "latest")
    .table("loan_silver.classified_loans")
)

# ─────────────────────────────────────────
# STEP 2: Filter High Risk Events
# ─────────────────────────────────────────

high_risk_df = (
    changes_df
    .filter(
        (col("dpd_bucket").isin("61-90", "90+")) |
        (col("is_npa") == True)
    )
    .select(
        "facility_id",
        "buyer_pan",
        "fi_name",
        "dpd_bucket",
        "days_overdue",
        "total_receivable",
        "is_npa",
        "updated_at"
    )
)

# ─────────────────────────────────────────
# STEP 3: Send Alerts for Each Batch
# ─────────────────────────────────────────

def send_alerts(batch_df, batch_id):
    
    if batch_df.count() == 0:
        return
    
    alerts = batch_df.collect()
    
    for alert in alerts:
        
        # NPA Alert — most severe
        if alert["is_npa"]:
            print(f"""
            🚨 NPA ALERT
            Buyer:       {alert['buyer_pan']}
            Facility:    {alert['facility_id']}
            Amount:      ₹{alert['total_receivable']:,.2f}
            Lender:      {alert['fi_name']}
            Time:        {alert['updated_at']}
            """)

        # 90+ DPD Alert
        elif alert["dpd_bucket"] == "90+":
            print(f"""
            ⚠️ 90+ DPD ALERT
            Buyer:       {alert['buyer_pan']}
            Days Overdue:{alert['days_overdue']}
            Amount:      ₹{alert['total_receivable']:,.2f}
            """)

        # 61-90 DPD Warning
        elif alert["dpd_bucket"] == "61-90":
            print(f"""
            ⚡ 61-90 DPD WARNING
            Buyer:       {alert['buyer_pan']}
            Days Overdue:{alert['days_overdue']}
            Amount:      ₹{alert['total_receivable']:,.2f}
            """)

    # Save alerts to Delta for audit trail
    (
        batch_df
        .withColumn("alert_generated_at", current_timestamp())
        .write
        .format("delta")
        .mode("append")
        .saveAsTable("loan_gold.high_risk_alerts")
    )

# ─────────────────────────────────────────
# STEP 4: Start Alert Stream
# ─────────────────────────────────────────

alert_query = (
    high_risk_df.writeStream
    .foreachBatch(send_alerts)
    .option("checkpointLocation",
            "/Volumes/dgs/default/cricket_api_project/loan_events/loan_monitoring/checkpoints/alerts/")
    .trigger(availableNow=True)
    .start()
)

print("✅ Alert pipeline started")